In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# # Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# # Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# # kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install -q "transformers==4.46.3" "huggingface_hub==0.26.2" "datasets==3.0.2" "accelerate==1.1.1" "peft==0.13.2" "evaluate==0.4.3" sqlparse sentencepiece

# transformers — loads CodeT5-base and handles tokenization/training
# datasets — loads WikiSQL directly from Hugging Face
# peft — implements LoRA on top of the base model
# accelerate — handles device placement (GPU) cleanly
# evaluate — for computing accuracy metrics later
# sqlparse — useful later for validating/formatting generated SQL

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 672.0 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 28.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.5/447.5 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.7/472.7 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 79.6 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
big

In [3]:
# verify the GPU is detected
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

True Tesla T4


In [4]:
# Load Dataset
from datasets import load_dataset

dataset = load_dataset("Salesforce/wikisql", revision="refs/convert/parquet")
print(dataset)

0000.parquet:   0%|          | 0.00/25.2M [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/3.63M [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/7.71M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['phase', 'question', 'table', 'sql'],
        num_rows: 56355
    })
    validation: Dataset({
        features: ['phase', 'question', 'table', 'sql'],
        num_rows: 8421
    })
    test: Dataset({
        features: ['phase', 'question', 'table', 'sql'],
        num_rows: 15878
    })
})


In [5]:
# single raw example from the dataset
print(dataset["train"][0])

{'phase': 1, 'question': 'Tell me what the notes are for South Australia ', 'table': {'header': ['State/territory', 'Text/background colour', 'Format', 'Current slogan', 'Current series', 'Notes'], 'page_title': '', 'page_id': '', 'types': ['text', 'text', 'text', 'text', 'text', 'text'], 'id': '1-1000181-1', 'section_title': '', 'caption': '', 'rows': [['Australian Capital Territory', 'blue/white', 'Yaa·nna', 'ACT · CELEBRATION OF A CENTURY 2013', 'YIL·00A', 'Slogan screenprinted on plate'], ['New South Wales', 'black/yellow', 'aa·nn·aa', 'NEW SOUTH WALES', 'BX·99·HI', 'No slogan on current series'], ['New South Wales', 'black/white', 'aaa·nna', 'NSW', 'CPX·12A', 'Optional white slimline series'], ['Northern Territory', 'ochre/white', 'Ca·nn·aa', 'NT · OUTBACK AUSTRALIA', 'CB·06·ZZ', 'New series began in June 2011'], ['Queensland', 'maroon/white', 'nnn·aaa', 'QUEENSLAND · SUNSHINE STATE', '999·TLG', 'Slogan embossed on plate'], ['South Australia', 'black/white', 'Snnn·aaa', 'SOUTH AUS

In [6]:
# print human readable SQL from the first raw example of the dataset
print(dataset["train"][0]["sql"]["human_readable"])

SELECT Notes FROM table WHERE Current slogan = SOUTH AUSTRALIA


In [7]:
# SQL reconstruction function, Since SQL queries in current dataset has quality issues from execution perspective
AGG_OPS = ['', 'MAX', 'MIN', 'COUNT', 'SUM', 'AVG']
COND_OPS = ['=', '>', '<']
def reconstruct_sql(example):
    header = example['table']['header']
    types = example['table']['types']
    sql = example['sql']

    sel_col = header[sql['sel']]
    agg = AGG_OPS[sql['agg']]

    # Quote column name if it has spaces/special chars
    def quote_col(col):
        return f'"{col}"'

    select_clause = f"SELECT {agg}({quote_col(sel_col)})" if agg else f"SELECT {quote_col(sel_col)}"

    conditions = []
    for col_idx, op_idx, value in zip(
        sql['conds']['column_index'],
        sql['conds']['operator_index'],
        sql['conds']['condition']
    ):
        col_name = quote_col(header[col_idx])
        op = COND_OPS[op_idx]
        col_type = types[col_idx]

        # Quote value only if it's not purely numeric
# matching the literal's format to the column's actual type avoids relying on the database engine's implicit type coercion 
# (SQLite is lenient about this, but stricter SQL engines are not), so the generated queries stay portable and 
# behave predictably regardless of which value is being compared.
        val = value if col_type == 'real' else f"'{value}'"

        conditions.append(f"{col_name} {op} {val}")

    where_clause = f" WHERE {' AND '.join(conditions)}" if conditions else ""
    return f"{select_clause} FROM table{where_clause}"

In [8]:
# Testing reconstruction function
example = dataset["train"][0]
print("Original (messy):", example['sql']['human_readable'])
print("Reconstructed:   ", reconstruct_sql(example))

Original (messy): SELECT Notes FROM table WHERE Current slogan = SOUTH AUSTRALIA
Reconstructed:    SELECT "Notes" FROM table WHERE "Current slogan" = 'SOUTH AUSTRALIA'


In [9]:
# each WikiSQL example has its own table with different columns, the model needs to see the column names as part of its input — it can't guess what's queryable from the question alone. The standard convention for T5-style text-to-SQL fine-tuning is to serialize this as a single input string like:
# translate to SQL: <question> | columns: <col1>, <col2>, <col3>, ...

In [10]:
# Builds training pairs which feeds the question along with the columns of that table to the model as an input string
def build_training_pair(example):
    question = example['question'].strip()
    columns = example['table']['header']
    columns_str = ", ".join(columns)

    input_text = f"translate to SQL: {question} | columns: {columns_str}"
    target_text = reconstruct_sql(example)

    return {"input_text": input_text, "target_text": target_text}

In [11]:
# Tests pair building of the input question along with table details(columns) and also output query
pair = build_training_pair(dataset["train"][0])
print("INPUT: ", pair["input_text"])
print("TARGET:", pair["target_text"])

INPUT:  translate to SQL: Tell me what the notes are for South Australia | columns: State/territory, Text/background colour, Format, Current slogan, Current series, Notes
TARGET: SELECT "Notes" FROM table WHERE "Current slogan" = 'SOUTH AUSTRALIA'


In [12]:
# applies the build_training_pair function to every row across all three splits (train, validation, test) and returns a new dataset with the results.
# .map(fn) takes a function and runs it once per example (each row)
# The return value gets merged into each row, not replacing it — that's why, before we ran remove_columns, each row still had phase, question, table, sql plus the two new keys input_text and target_text. 
# .map() adds whatever keys your function returns onto the existing row.
# It's not a Python for loop under the hood — Hugging Face's datasets library processes this efficiently in batches and caches the result to disk, 
# which is why it can run over 56k+ rows without loading everything into memory at once or being painfully slow.

processed = dataset.map(build_training_pair)

# remove unneeded columns
processed = processed.remove_columns(['phase', 'question', 'table', 'sql'])
print(processed["train"][0])

Map:   0%|          | 0/56355 [00:00<?, ? examples/s]

Map:   0%|          | 0/8421 [00:00<?, ? examples/s]

Map:   0%|          | 0/15878 [00:00<?, ? examples/s]

{'input_text': 'translate to SQL: Tell me what the notes are for South Australia | columns: State/territory, Text/background colour, Format, Current slogan, Current series, Notes', 'target_text': 'SELECT "Notes" FROM table WHERE "Current slogan" = \'SOUTH AUSTRALIA\''}


In [13]:
# This searches for a text column containing something that looks numeric
# numeric-looking values (1998, 183, 10.73, etc.) is now properly quoted as a string ('1998', '183') because their column type is text, not real. 
# We convert them to quoted strings because we're now basing the decision on the column's declared type in the table schema (text vs real), 
# not on how the individual value happens to look.
for i in range(200):
    ex = dataset["train"][i]
    for col_idx, value in zip(ex['sql']['conds']['column_index'], ex['sql']['conds']['condition']):
        if ex['table']['types'][col_idx] == 'text':
            try:
                float(value)
                print(f"Row {i}: text column with numeric-looking value: {value}")
                print("Reconstructed:", reconstruct_sql(ex))
                break
            except ValueError:
                continue

Row 6: text column with numeric-looking value: 1998
Reconstructed: SELECT "Manufacturer" FROM table WHERE "Order Year" = '1998'
Row 9: text column with numeric-looking value: 2000
Reconstructed: SELECT "Powertrain (Engine/Transmission)" FROM table WHERE "Order Year" = '2000'
Row 12: text column with numeric-looking value: 6
Reconstructed: SELECT "School/Club Team" FROM table WHERE "No." = '6'
Row 35: text column with numeric-looking value: 2004
Reconstructed: SELECT "Position" FROM table WHERE "Years in Toronto" = '2004'
Row 39: text column with numeric-looking value: 117
Reconstructed: SELECT "Best finish" FROM table WHERE "Scoring rank" = '117'
Row 41: text column with numeric-looking value: 183
Reconstructed: SELECT COUNT("Wins") FROM table WHERE "Money list rank" = '183'
Row 46: text column with numeric-looking value: 1
Reconstructed: SELECT COUNT("Team") FROM table WHERE "Draft Pick #" = '1'
Row 47: text column with numeric-looking value: 10
Reconstructed: SELECT "Position" FROM t

In [14]:
# TOKENIZATION

In [15]:
!pip install -q "transformers==4.46.3" sentencepiece

In [16]:
# Load the CodeT5 tokenizer
from transformers import AutoTokenizer

model_name = "Salesforce/codet5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [17]:
text = "select name from employees"
tokens = tokenizer(text)
print(tokens)

{'input_ids': [1, 4025, 508, 628, 24539, 25521, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}


In [18]:
# pulls input_text and target_text fields from first 1000 training rows
sample_inputs = processed["train"]["input_text"][:1000]
sample_targets = processed["train"]["target_text"][:1000]

# tokenizer.encode(t) — converts a text string into its list of token IDs (the numerical form CodeT5 actually reads). 
# This is different from character or word count — a tokenizer often splits words into sub-word pieces, so token count doesn't map 1:1 to word count.

# The number of tokens that input/target string produces stored in a list
input_lens = [len(tokenizer.encode(t)) for t in sample_inputs]
target_lens = [len(tokenizer.encode(t)) for t in sample_targets]

# from the list of token lengths for input/target strings find max and avg token lengths
print("Input lengths  — max:", max(input_lens), "avg:", sum(input_lens)/len(input_lens))
print("Target lengths — max:", max(target_lens), "avg:", sum(target_lens)/len(target_lens))

Input lengths  — max: 128 avg: 54.356
Target lengths — max: 90 avg: 24.907


In [19]:
# finds token count of input/target strings(all of the 56,355 rows not just first 1000 training rows in the dataset)
input_lens_full = [len(tokenizer.encode(t)) for t in processed["train"]["input_text"]]
target_lens_full = [len(tokenizer.encode(t)) for t in processed["train"]["target_text"]]

# Finds max token count from all the input/target strings 
print("Full input lengths  — max:", max(input_lens_full))
print("Full target lengths — max:", max(target_lens_full))

Full input lengths  — max: 331
Full target lengths — max: 191


In [20]:
import numpy as np

for p in [90, 95, 99, 100]:
    print(f"Input  p{p}:", np.percentile(input_lens_full, p))
    print(f"Target p{p}:", np.percentile(target_lens_full, p))

Input  p90: 66.0
Target p90: 36.0
Input  p95: 74.0
Target p95: 41.0
Input  p99: 101.0
Target p99: 51.0
Input  p100: 331.0
Target p100: 191.0


In [21]:
# he jump from p99 (101/51) to p100 (331/191) shows less than 1% of examples are extreme outliers, 
# while 99% of real data comfortably fits under ~101 input / ~51 target tokens.
# we consider 128 tokens for input and 64 tokens for output based on the analysis done

In [22]:
# For T5-style models, input_ids come from the input text, and labels come from the target text 
def tokenize_function(example):
    model_inputs = tokenizer(
        example["input_text"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        example["target_text"],
        max_length=64,
        truncation=True,
        padding="max_length"
    )

    label_ids = labels["input_ids"]
# need to replace padding token IDs in labels with -100,since that's the value PyTorch's loss function is told to ignore (otherwise the model would be penalized for "failing" to predict padding, which isn't a real learning signal):
    label_ids = [(id_ if id_ != tokenizer.pad_token_id else -100) for id_ in label_ids]
# Adds this cleaned-up list into the model_inputs dict under the key "labels", alongside input_ids and attention_mask
    model_inputs["labels"] = label_ids

    return model_inputs

In [23]:
# Apply tokenization across all splits, and remove the now-unneeded text columns
tokenized = processed.map(tokenize_function, batched=False)
tokenized = tokenized.remove_columns(["input_text", "target_text"])
print(tokenized)
print(tokenized["train"][0])

Map:   0%|          | 0/56355 [00:00<?, ? examples/s]

Map:   0%|          | 0/8421 [00:00<?, ? examples/s]

Map:   0%|          | 0/15878 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 56355
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 8421
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 15878
    })
})
{'input_ids': [1, 13929, 358, 3063, 30, 29860, 1791, 4121, 326, 10913, 854, 364, 348, 15347, 432, 27008, 287, 1155, 571, 2168, 30, 3287, 19, 88, 25313, 16, 3867, 19, 9342, 15046, 16, 4077, 16, 6562, 272, 1330, 304, 16, 6562, 4166, 16, 29584, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

In [24]:
# Set the format to PyTorch tensors, since the Trainer we'll use next expects tensors, not plain lists
tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [25]:
import huggingface_hub, transformers, peft
print("huggingface_hub:", huggingface_hub.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)

huggingface_hub: 0.26.2
transformers: 4.46.3
peft: 0.13.2


In [26]:
# Load the pretrained CodeT5-base model
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/892M [00:00<?, ?B/s]

In [27]:
print(type(model))

<class 'transformers.models.t5.modeling_t5.T5ForConditionalGeneration'>


In [28]:
# test the complete tokenizer + model
# output might not be so good yet. Salesforce/codet5-base is a pretrained CodeT5 model, but eventual goal is to fine-tune it on WikiSQL so that it learns natural-language → SQL task.
text = "translate English to SQL: show me all employees"

inputs = tokenizer(text, return_tensors="pt")

outputs = model.generate(**inputs, max_length=128)

result = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(result)

 public static void


In [29]:
# Define the LoRA configuration — rank 16, attention layers only
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM, # tells PEFT this is an encoder-decoder generation task (not classification or causal LM), which affects how it wraps the model.
    r=16,
    lora_alpha=32,  # setting alpha to 2× the rank
    target_modules=["q", "v"],  # CodeT5's internal names for the query and value attention projections — this is what "attention layers only" translates to in code.
    lora_dropout=0.05,
    bias="none"
)

In [30]:
# Wrap the base model with LoRA and check how many parameters are actually trainable
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,769,472 || all params: 224,651,520 || trainable%: 0.7877


In [31]:
!pip install -q wandb

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [32]:
# import wandb
# wandb.login()

In [33]:
from kaggle_secrets import UserSecretsClient
import wandb

user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("WANDB_API_KEY")
wandb.login(key=wandb_key)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: siddharthchaudhari40 (ssvbtech) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [34]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

In [35]:
import os
os.environ["WANDB_PROJECT"] = "nl-to-sql-codet5"

from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

training_args = TrainingArguments(
    output_dir="./results/run1_ep3_bs8_lr2e-4",
    run_name="run1_ep3_bs8_lr2e-4",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-4,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=50,
    report_to="wandb",
    fp16=True,
    push_to_hub=True,
    hub_model_id="siddharth57/codet5-wikisql-run1",
    hub_strategy="every_save",
)

In [36]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
)

In [37]:
# trainer.train()
# trainer.push_to_hub()

In [42]:
from peft import PeftModel

base = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model = PeftModel.from_pretrained(base, "siddharth57/codet5-wikisql-run1")
# model.eval()
# model.to(device)

In [43]:
# Evaluate on Test Set
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

PeftModelForSeq2SeqLM(
  (base_model): LoraModel(
    (model): T5ForConditionalGeneration(
      (shared): Embedding(32100, 768)
      (encoder): T5Stack(
        (embed_tokens): Embedding(32100, 768)
        (block): ModuleList(
          (0): T5Block(
            (layer): ModuleList(
              (0): T5LayerSelfAttention(
                (SelfAttention): T5Attention(
                  (q): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=False)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.05, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=768, out_features=16, bias=False)
                    )
                    (lora_B): ModuleDict(
                      (default): Linear(in_features=16, out_features=768, bias=False)
                    )
                    (lora_embedding_A): ParameterDict()
            

In [44]:
sample = tokenized["test"].select(range(5))

for i in range(5):
    input_ids = sample["input_ids"][i].unsqueeze(0).to(device)
    attention_mask = sample["attention_mask"][i].unsqueeze(0).to(device)

    with torch.no_grad():
        output = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_length=64)

    predicted_sql = tokenizer.decode(output[0], skip_special_tokens=True)
    actual_sql = processed["test"][i]["target_text"]
    question = processed["test"][i]["input_text"]

    print("INPUT:    ", question)
    print("PREDICTED:", predicted_sql)
    print("ACTUAL:   ", actual_sql)
    print("---")

INPUT:     translate to SQL: What is terrence ross' nationality | columns: Player, No., Nationality, Position, Years in Toronto, School/Club Team
PREDICTED: SELECT "Nationality" FROM table WHERE "Player" = 'terrence ross'
ACTUAL:    SELECT "Nationality" FROM table WHERE "Player" = 'Terrence Ross'
---
INPUT:     translate to SQL: What clu was in toronto 1995-96 | columns: Player, No., Nationality, Position, Years in Toronto, School/Club Team
PREDICTED: SELECT "School/Club Team" FROM table WHERE "Years in Toronto" = '1995-96'
ACTUAL:    SELECT "School/Club Team" FROM table WHERE "Years in Toronto" = '1995-96'
---
INPUT:     translate to SQL: which club was in toronto 2003-06 | columns: Player, No., Nationality, Position, Years in Toronto, School/Club Team
PREDICTED: SELECT "School/Club Team" FROM table WHERE "Years in Toronto" = '2003-06'
ACTUAL:    SELECT "School/Club Team" FROM table WHERE "Years in Toronto" = '2003-06'
---
INPUT:     translate to SQL: how many schools or teams had jal

In [45]:
from tqdm import tqdm

In [46]:
# Run full test-set evaluation

model.eval()
predictions = []
actuals = []

batch_size = 32
test_data = tokenized["test"]
test_processed = processed["test"]

for i in tqdm(range(0, len(test_data), batch_size)):
    batch = test_data[i:i+batch_size]
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_length=64)

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    predictions.extend(decoded)
    actuals.extend(test_processed["target_text"][i:i+batch_size])

print(f"Generated {len(predictions)} predictions")

100%|██████████| 497/497 [11:36<00:00,  1.40s/it]

Generated 15878 predictions


In [47]:
# compute both accuracy numbers
def normalize(sql):
    return sql.strip()

strict_matches = sum(1 for p, a in zip(predictions, actuals) if normalize(p) == normalize(a))
ci_matches = sum(1 for p, a in zip(predictions, actuals) if normalize(p).lower() == normalize(a).lower())

strict_accuracy = strict_matches / len(actuals) * 100
ci_accuracy = ci_matches / len(actuals) * 100

print(f"Strict exact-match accuracy: {strict_accuracy:.2f}%")
print(f"Case-insensitive exact-match accuracy: {ci_accuracy:.2f}%")

Strict exact-match accuracy: 58.32%
Case-insensitive exact-match accuracy: 64.45%


In [48]:
# Set up and start run 2 (5 epochs, same batch size and LR)

In [49]:
model2 = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model2 = get_peft_model(model2, lora_config)
model2.print_trainable_parameters()

trainable params: 1,769,472 || all params: 224,651,520 || trainable%: 0.7877


In [50]:
training_args2 = TrainingArguments(
    output_dir="./results/run2_ep7_bs8_lr2e-4",
    run_name="run2_ep7_bs8_lr2e-4",
    num_train_epochs=7,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-4,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=50,
    report_to="wandb",
    fp16=True,
    push_to_hub=True,
    hub_model_id="siddharth57/codet5-wikisql-run2",
    hub_strategy="every_save",
)

In [51]:
data_collator2 = DataCollatorForSeq2Seq(tokenizer, model=model2)

trainer2 = Trainer(
    model=model2,
    args=training_args2,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator2,
)

In [52]:
# trainer2.train()
# trainer2.push_to_hub()

In [53]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [54]:
from peft import PeftModel

base = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model2 = PeftModel.from_pretrained(base, "siddharth57/codet5-wikisql-run2")
model2.eval()
model2.to(device)

adapter_config.json:   0%|          | 0.00/641 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/7.10M [00:00<?, ?B/s]

PeftModelForSeq2SeqLM(
  (base_model): LoraModel(
    (model): T5ForConditionalGeneration(
      (shared): Embedding(32100, 768)
      (encoder): T5Stack(
        (embed_tokens): Embedding(32100, 768)
        (block): ModuleList(
          (0): T5Block(
            (layer): ModuleList(
              (0): T5LayerSelfAttention(
                (SelfAttention): T5Attention(
                  (q): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=False)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.05, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=768, out_features=16, bias=False)
                    )
                    (lora_B): ModuleDict(
                      (default): Linear(in_features=16, out_features=768, bias=False)
                    )
                    (lora_embedding_A): ParameterDict()
            

In [55]:
batch_size = 32
test_data = tokenized["test"]
test_processed = processed["test"]

In [56]:
def normalize(sql):
    return sql.strip()

In [59]:
predictions2 = []
actuals2 = []

for i in tqdm(range(0, len(test_data), batch_size)):
    batch = test_data[i:i+batch_size]
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model2.generate(input_ids=input_ids, attention_mask=attention_mask, max_length=64)

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    predictions2.extend(decoded)
    actuals2.extend(test_processed["target_text"][i:i+batch_size])

strict_matches2 = sum(1 for p, a in zip(predictions2, actuals2) if normalize(p) == normalize(a))
ci_matches2 = sum(1 for p, a in zip(predictions2, actuals2) if normalize(p).lower() == normalize(a).lower())

print(f"Run 2 — Strict exact-match accuracy: {strict_matches2/len(actuals2)*100:.2f}%")
print(f"Run 2 — Case-insensitive exact-match accuracy: {ci_matches2/len(actuals2)*100:.2f}%")

100%|██████████| 497/497 [11:32<00:00,  1.39s/it]

Run 2 — Strict exact-match accuracy: 63.47%
Run 2 — Case-insensitive exact-match accuracy: 68.76%


In [60]:
import sqlite3

def create_temp_db(example):
    conn = sqlite3.connect(":memory:")
    cursor = conn.cursor()

    header = example['table']['header']
    types = example['table']['types']
    rows = example['table']['rows']

    # Map WikiSQL types to SQLite types
    col_defs = []
    for col, col_type in zip(header, types):
        sqlite_type = "REAL" if col_type == "real" else "TEXT"
        col_defs.append(f'"{col}" {sqlite_type}')

    create_stmt = f'CREATE TABLE "table" ({", ".join(col_defs)})'
    cursor.execute(create_stmt)

    placeholders = ", ".join(["?"] * len(header))
    insert_stmt = f'INSERT INTO "table" VALUES ({placeholders})'
    cursor.executemany(insert_stmt, rows)

    conn.commit()
    return conn

In [61]:
example = dataset["test"][0]
conn = create_temp_db(example)
cursor = conn.cursor()
cursor.execute('SELECT * FROM "table" LIMIT 3')
print(cursor.fetchall())
conn.close()

[('Aleksandar Radojević', '25', 'Serbia', 'Center', '1999-2000', 'Barton CC (KS)'), ('Shawn Respert', '31', 'United States', 'Guard', '1997-98', 'Michigan State'), ('Quentin Richardson', 'N/A', 'United States', 'Forward', '2013-present', 'DePaul')]


In [62]:
import re

def make_executable(sql):
    sql = re.sub(r'\bFROM table\b', 'FROM "table"', sql)
    # Make string equality comparisons case-insensitive
    sql = re.sub(r"(=\s*'[^']*')", r"\1 COLLATE NOCASE", sql)
    return sql

def execution_match(example, predicted_sql):
    try:
        conn = create_temp_db(example)
        cursor = conn.cursor()

        actual_sql = reconstruct_sql(example)

        cursor.execute(make_executable(predicted_sql))
        predicted_result = set(cursor.fetchall())

        cursor.execute(make_executable(actual_sql))
        actual_result = set(cursor.fetchall())

        conn.close()
        return predicted_result == actual_result

    except Exception:
        return False

In [63]:
test_examples = dataset["test"]

In [64]:
for i in range(5):
    example = test_examples[i]
    predicted_sql = predictions2[i]
    match = execution_match(example, predicted_sql)

    print("PREDICTED:", predicted_sql)
    print("ACTUAL:   ", reconstruct_sql(example))
    print("MATCH:    ", match)
    print("---")

PREDICTED: SELECT "Nationality" FROM table WHERE "Player" = 'terrence ross'
ACTUAL:    SELECT "Nationality" FROM table WHERE "Player" = 'Terrence Ross'
MATCH:     True
---
PREDICTED: SELECT "School/Club Team" FROM table WHERE "Years in Toronto" = '1995-96'
ACTUAL:    SELECT "School/Club Team" FROM table WHERE "Years in Toronto" = '1995-96'
MATCH:     True
---
PREDICTED: SELECT "School/Club Team" FROM table WHERE "Years in Toronto" = '2003-06'
ACTUAL:    SELECT "School/Club Team" FROM table WHERE "Years in Toronto" = '2003-06'
MATCH:     True
---
PREDICTED: SELECT COUNT("School/Club Team") FROM table WHERE "Player" = 'Jalen Rose'
ACTUAL:    SELECT COUNT("School/Club Team") FROM table WHERE "Player" = 'Jalen Rose'
MATCH:     True
---
PREDICTED: SELECT "Fastest Lap" FROM table WHERE "Circuit" = 'Assen'
ACTUAL:    SELECT "Round" FROM table WHERE "Circuit" = 'Assen'
MATCH:     False
---


In [65]:
matches = 0
total = len(test_examples)

for i in tqdm(range(total)):
    example = test_examples[i]
    predicted_sql = predictions2[i]
    if execution_match(example, predicted_sql):
        matches += 1

execution_accuracy = matches / total * 100
print(f"Run 2 — Execution accuracy: {execution_accuracy:.2f}%")

100%|██████████| 15878/15878 [00:08<00:00, 1805.03it/s]

Run 2 — Execution accuracy: 77.86%


In [69]:
def execution_match_debug(example, predicted_sql):
    conn = create_temp_db(example)
    cursor = conn.cursor()

    actual_sql = reconstruct_sql(example)

    cursor.execute(make_executable(predicted_sql))
    predicted_result = set(cursor.fetchall())

    cursor.execute(make_executable(actual_sql))
    actual_result = set(cursor.fetchall())

    conn.close()
    return predicted_result == actual_result

example = test_examples[1]
predicted_sql = predictions2[1]
print(execution_match_debug(example, predicted_sql))

True


In [70]:
base1 = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model1 = PeftModel.from_pretrained(base1, "siddharth57/codet5-wikisql-run1")
model1.eval()
model1.to(device)

PeftModelForSeq2SeqLM(
  (base_model): LoraModel(
    (model): T5ForConditionalGeneration(
      (shared): Embedding(32100, 768)
      (encoder): T5Stack(
        (embed_tokens): Embedding(32100, 768)
        (block): ModuleList(
          (0): T5Block(
            (layer): ModuleList(
              (0): T5LayerSelfAttention(
                (SelfAttention): T5Attention(
                  (q): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=False)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.05, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=768, out_features=16, bias=False)
                    )
                    (lora_B): ModuleDict(
                      (default): Linear(in_features=16, out_features=768, bias=False)
                    )
                    (lora_embedding_A): ParameterDict()
            

In [71]:
predictions1 = []
actuals1 = []

for i in tqdm(range(0, len(test_data), batch_size)):
    batch = test_data[i:i+batch_size]
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model1.generate(input_ids=input_ids, attention_mask=attention_mask, max_length=64)

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    predictions1.extend(decoded)
    actuals1.extend(test_processed["target_text"][i:i+batch_size])

100%|██████████| 497/497 [11:58<00:00,  1.45s/it]


In [72]:
matches1 = 0
total = len(test_examples)

for i in tqdm(range(total)):
    example = test_examples[i]
    predicted_sql = predictions1[i]
    if execution_match(example, predicted_sql):
        matches1 += 1

execution_accuracy1 = matches1 / total * 100
print(f"Run 1 — Execution accuracy: {execution_accuracy1:.2f}%")

100%|██████████| 15878/15878 [00:09<00:00, 1758.30it/s]

Run 1 — Execution accuracy: 74.15%


In [73]:
# QUERY VALIDATION/GUARDRAILS

In [74]:
import re

DANGEROUS_KEYWORDS = ["INSERT", "UPDATE", "DELETE", "DROP", "ALTER", "CREATE", "TRUNCATE", "REPLACE", "ATTACH", "DETACH"]

def validate_sql(sql, header):
    sql_upper = sql.upper().strip()

    # Must start with SELECT
    if not sql_upper.startswith("SELECT"):
        return False, "Not a SELECT statement"

    # Block dangerous keywords anywhere in the query
    for kw in DANGEROUS_KEYWORDS:
        if re.search(rf'\b{kw}\b', sql_upper):
            return False, f"Contains disallowed keyword: {kw}"

    # Block statement chaining (a semicolon followed by more content)
    if re.search(r';\s*\S', sql):
        return False, "Multiple statements not allowed"

    # Check that referenced columns exist in the schema
    quoted_cols = re.findall(r'"([^"]+)"', sql)
    for col in quoted_cols:
        if col not in header and col != "table":
            return False, f"Unknown column referenced: {col}"

    return True, "Valid"

In [75]:
example = test_examples[0]
header = example['table']['header']

# A real prediction
print(validate_sql(predictions2[0], header))

# A deliberately dangerous example
print(validate_sql('DROP TABLE "table"; SELECT * FROM "table"', header))

# A hallucinated column example
print(validate_sql('SELECT "FakeColumn" FROM "table"', header))

(True, 'Valid')
(False, 'Not a SELECT statement')
(False, 'Unknown column referenced: FakeColumn')


In [76]:
def generate_with_correction(question, columns, header, max_retries=2):
    columns_str = ", ".join(columns)
    input_text = f"translate to SQL: {question} | columns: {columns_str}"

    for attempt in range(max_retries + 1):
        inputs = tokenizer(input_text, return_tensors="pt", max_length=128, truncation=True).to(device)

        with torch.no_grad():
            output = model2.generate(**inputs, max_length=64)

        predicted_sql = tokenizer.decode(output[0], skip_special_tokens=True)
        is_valid, reason = validate_sql(predicted_sql, header)

        if is_valid:
            return predicted_sql, attempt, True

        # Retry: append the failure reason to the input, nudging the model
        input_text = f"translate to SQL: {question} | columns: {columns_str} | previous attempt failed: {reason}, try again"

    return predicted_sql, max_retries, False  # exhausted retries, return last attempt

In [77]:
example = test_examples[5]
question = example['question']
columns = example['table']['header']
header = example['table']['header']

sql, attempts, success = generate_with_correction(question, columns, header)
print("SQL:", sql)
print("Attempts used:", attempts)
print("Success:", success)

SQL: SELECT COUNT("No") FROM table WHERE "Race winner" = 'Kevin Curtain'
Attempts used: 0
Success: True


In [78]:
# Directly test validate_sql's rejection + the retry prompt construction, without full generation
test_reason = "Unknown column referenced: FakeCol"
retry_input = f"translate to SQL: test question | columns: A, B, C | previous attempt failed: {test_reason}, try again"
print(retry_input)

inputs = tokenizer(retry_input, return_tensors="pt", max_length=128, truncation=True).to(device)
with torch.no_grad():
    output = model2.generate(**inputs, max_length=64)
print(tokenizer.decode(output[0], skip_special_tokens=True))

translate to SQL: test question | columns: A, B, C | previous attempt failed: Unknown column referenced: FakeCol, try again
SELECT "try again" FROM table WHERE "A" = 'test'


In [79]:
class QueryCache:
    def __init__(self):
        self.cache = {}
        self.hits = 0
        self.misses = 0

    def get(self, question, columns):
        key = (question.strip().lower(), tuple(columns))
        if key in self.cache:
            self.hits += 1
            return self.cache[key]
        self.misses += 1
        return None

    def set(self, question, columns, sql):
        key = (question.strip().lower(), tuple(columns))
        self.cache[key] = sql

    def stats(self):
        total = self.hits + self.misses
        hit_rate = (self.hits / total * 100) if total > 0 else 0
        return {"hits": self.hits, "misses": self.misses, "hit_rate": f"{hit_rate:.1f}%"}

query_cache = QueryCache()

In [80]:
def generate_with_cache(question, columns, header):
    cached = query_cache.get(question, columns)
    if cached is not None:
        return cached, True  # True = was a cache hit

    sql, attempts, success = generate_with_correction(question, columns, header)
    query_cache.set(question, columns, sql)
    return sql, False

In [81]:
example = test_examples[0]
question = example['question']
columns = example['table']['header']
header = example['table']['header']

sql1, hit1 = generate_with_cache(question, columns, header)
sql2, hit2 = generate_with_cache(question, columns, header)

print("First call  — SQL:", sql1, "| Cache hit:", hit1)
print("Second call — SQL:", sql2, "| Cache hit:", hit2)
print(query_cache.stats())

First call  — SQL: SELECT "Nationality" FROM table WHERE "Player" = 'terrence ross' | Cache hit: False
Second call — SQL: SELECT "Nationality" FROM table WHERE "Player" = 'terrence ross' | Cache hit: True
{'hits': 1, 'misses': 1, 'hit_rate': '50.0%'}


In [82]:
!pip install -q bentoml

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 4.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.8/50.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.7/221.7 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.4/118.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 7.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
datasets 3.0.2 requires fsspec[http]<=2024.9.0,>=2023.1.0, but you have fsspe

In [83]:
import datasets
print(datasets.__version__)

# Quick functional check — not a full reload, just confirm the library still operates
from datasets import Dataset
test = Dataset.from_dict({"a": [1, 2, 3]})
print(test)

3.0.2
Dataset({
    features: ['a'],
    num_rows: 3
})


In [84]:
%%writefile service.py
import re
import torch
import bentoml
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel

DANGEROUS_KEYWORDS = ["INSERT", "UPDATE", "DELETE", "DROP", "ALTER", "CREATE", "TRUNCATE", "REPLACE", "ATTACH", "DETACH"]

def validate_sql(sql, header):
    sql_upper = sql.upper().strip()

    if not sql_upper.startswith("SELECT"):
        return False, "Not a SELECT statement"

    for kw in DANGEROUS_KEYWORDS:
        if re.search(rf'\b{kw}\b', sql_upper):
            return False, f"Contains disallowed keyword: {kw}"

    if re.search(r';\s*\S', sql):
        return False, "Multiple statements not allowed"

    quoted_cols = re.findall(r'"([^"]+)"', sql)
    for col in quoted_cols:
        if col not in header and col != "table":
            return False, f"Unknown column referenced: {col}"

    return True, "Valid"


@bentoml.service
class NLToSQLService:

    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.tokenizer = AutoTokenizer.from_pretrained("Salesforce/codet5-base")

        base = AutoModelForSeq2SeqLM.from_pretrained("Salesforce/codet5-base")
        self.model = PeftModel.from_pretrained(base, "siddharth57/codet5-wikisql-run2")
        self.model.eval()
        self.model.to(self.device)

        self.cache = {}

    def _generate(self, question, columns):
        columns_str = ", ".join(columns)
        input_text = f"translate to SQL: {question} | columns: {columns_str}"

        inputs = self.tokenizer(input_text, return_tensors="pt", max_length=128, truncation=True).to(self.device)

        with torch.no_grad():
            output = self.model.generate(**inputs, max_length=64)

        return self.tokenizer.decode(output[0], skip_special_tokens=True)

    @bentoml.api
    def query(self, question: str, columns: list[str]) -> dict:
        cache_key = (question.strip().lower(), tuple(columns))

        if cache_key in self.cache:
            return {"sql": self.cache[cache_key], "cached": True, "valid": True}

        sql = self._generate(question, columns)
        is_valid, reason = validate_sql(sql, columns)

        result = {"sql": sql, "cached": False, "valid": is_valid}
        if not is_valid:
            result["reason"] = reason
        else:
            self.cache[cache_key] = sql

        return result

Overwriting service.py


In [85]:
import importlib
import service as svc
importlib.reload(svc)

test_service = svc.NLToSQLService()

result = test_service.query(
    question="What is terrence ross' nationality",
    columns=["Player", "No.", "Nationality", "Position", "Years in Toronto", "School/Club Team"]
)
print(result)

# Call again with the same inputs to confirm caching works
result2 = test_service.query(
    question="What is terrence ross' nationality",
    columns=["Player", "No.", "Nationality", "Position", "Years in Toronto", "School/Club Team"]
)
print(result2)

{'sql': 'SELECT "Nationality" FROM table WHERE "Player" = \'terrence ross\'', 'cached': False, 'valid': True}
{'sql': 'SELECT "Nationality" FROM table WHERE "Player" = \'terrence ross\'', 'cached': True, 'valid': True}


In [86]:
import random

random.seed(42)
demo_tables = {}
candidate_indices = random.sample(range(len(test_examples)), 30)  # oversample, then take first 5 unique

for idx in candidate_indices:
    example = test_examples[idx]
    table_name = example['table']['name'] or f"table_idx_{idx}"

    if table_name not in demo_tables:
        demo_tables[table_name] = example

    if len(demo_tables) == 5:
        break

for name, ex in demo_tables.items():
    print(f"{name}: columns = {ex['table']['header']}")
    print(f"  Sample question: {ex['question']}")
    print()

table_idx_10476: columns = ['Team', 'Head Coach', 'President', 'Home Ground', 'Location']
  Sample question: Who is the Head Coach of the team whose President is Mario Volarevic?

table_18143210_2: columns = ['Club', 'First season in top division', 'Number of seasons in top division', 'First season of current spell in top division', 'Number of seasons in Liga MX', 'Top division titles']
  Sample question: How many 'number of seasons in top division' were played if the 'first season in top division' games is in 1990-91?

table_12032893_1: columns = ['Name', '#', 'Position', 'Height', 'Weight', 'Year', 'Home Town', 'High School']
  Sample question: What height was the forward position at Crockett High School?

table_idx_12149: columns = ['Team 1', 'Agg.', 'Team 2', '1st leg', '2nd leg']
  Sample question: What is the score for the 2nd leg when Belasica is team 2?

table_idx_4506: columns = ['Name', 'Pole Position', 'Fastest Lap', 'Winning driver', 'Winning team', 'Report']
  Sample quest

In [87]:
!pip install -q gradio

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 3.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 30.5 MB/s eta 0:00:0000:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
tokenizers 0.20.3 requires huggingface-hub<1.0,>=0.16.4, but you have huggingface-hub 1.33.0 which is incompatible.
transformers 4.46.3 requires huggingface-hub<1.0,>=0.23.2, but you have huggingface-hub 1.33.0 which is incompatible.
datasets 3.0.2 requires fsspec[http]<=2024.9.0,>=2023.1.0, but you have fsspec 2026.9.0 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.45.0 which is incompatible.
google-adk 

In [88]:
example = demo_tables['table_idx_10476']
conn = create_temp_db(example)
cursor = conn.cursor()
sql = 'SELECT "Head Coach" FROM table WHERE "President" = \'mario volarevic\''
cursor.execute(make_executable(sql))
print(cursor.fetchall())
conn.close()

[('Steve Radoslavic',)]


In [89]:
import gradio as gr
import sqlite3

def run_demo_query(table_choice, question):
    example = demo_tables[table_choice]
    columns = example['table']['header']

    # Generate SQL using the same service logic
    result = test_service.query(question=question, columns=columns)
    sql = result['sql']
    is_valid = result['valid']

    if not is_valid:
        return sql, "❌ Invalid", f"Query rejected: {result.get('reason', 'unknown')}"

    # Execute against real table data
    try:
        conn = create_temp_db(example)
        cursor = conn.cursor()
        cursor.execute(make_executable(sql))
        rows = cursor.fetchall()
        conn.close()
        result_str = "\n".join(str(r) for r in rows) if rows else "(no rows returned)"
        return sql, "✅ Valid", result_str
    except Exception as e:
        return sql, "⚠️ Valid SQL, execution error", str(e)


def show_table_preview(table_choice):
    example = demo_tables[table_choice]
    header = example['table']['header']
    rows = example['table']['rows'][:5]
    preview = " | ".join(header) + "\n" + "\n".join(" | ".join(str(c) for c in row) for row in rows)
    return preview, example['question']


with gr.Blocks(title="NL to SQL Demo") as demo:
    gr.Markdown("# Natural Language → SQL\nFine-tuned CodeT5 on WikiSQL — pick a table, ask a question.")

    table_dropdown = gr.Dropdown(choices=list(demo_tables.keys()), label="Select a table", value=list(demo_tables.keys())[0])
    table_preview = gr.Textbox(label="Table preview (first 5 rows)", lines=6, interactive=False)

    question_input = gr.Textbox(label="Ask a question about this table", placeholder="e.g. Who is the head coach of...?")
    sample_question_btn = gr.Button("Use sample question")

    submit_btn = gr.Button("Generate SQL & Run", variant="primary")

    sql_output = gr.Textbox(label="Generated SQL")
    validity_output = gr.Textbox(label="Validation status")
    result_output = gr.Textbox(label="Query result", lines=5)

    table_dropdown.change(show_table_preview, inputs=table_dropdown, outputs=[table_preview, question_input])
    sample_question_btn.click(lambda t: demo_tables[t]['question'], inputs=table_dropdown, outputs=question_input)
    submit_btn.click(run_demo_query, inputs=[table_dropdown, question_input], outputs=[sql_output, validity_output, result_output])

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://8c885f5ceca10e6f02.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
